In [ ]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F

landing_root = 'abfss://raw@<account>.dfs.core.windows.net/azsql/saleslt'
bronze_root = 'abfss://<bronze-container>@<account>.dfs.core.windows.net/azsql/saleslt'
checkpoint_root = 'abfss://<checkpoint-container>@<account>.dfs.core.windows.net/landing_to_bronze/azsql/saleslt'
bronze_database = 'adventureworks_bronze'

In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.sql(f'CREATE DATABASE IF NOT EXISTS {bronze_database}')

for folder in mssparkutils.fs.ls(landing_root):
    if not folder.isDir:
        continue

    table_name = folder.name.rstrip('/')
    target_table = f'{bronze_database}.{table_name}'
    source_path = folder.path

    print(f'Processing table: {target_table}', flush=True)

    if spark.catalog.tableExists(target_table):
        source_schema = spark.table(target_table).drop('_ingested_at', '_source_file').schema
    else:
        source_schema = (
            spark.read.option('recursiveFileLookup', 'true')
            .option('pathGlobFilter', '*.parquet')
            .parquet(source_path).schema
        )

    bronze = (
        spark.readStream.format('parquet').schema(source_schema)
        .option('recursiveFileLookup', 'true')
        .option('pathGlobFilter', '*.parquet')
        .option('maxFilesPerTrigger', 100)
        .load(source_path)
        .withColumn('_ingested_at', F.current_timestamp())
        .withColumn('_source_file', F.input_file_name())
    )

    query = (
        bronze.writeStream.format('delta').outputMode('append')
        .option('path', f'{bronze_root}/{table_name}')
        .option('checkpointLocation', f'{checkpoint_root}/{table_name}')
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    query.awaitTermination()
    print(f'Completed table: {target_table}', flush=True)